# HyperKT Objective 01 — Stage 8A: Source Audit + Controlled Ablations
**Runtime:** Google Colab **T4 GPU** (L4 optional). **Inputs:** `hyperkt_stage2.zip`, `hyperkt_stage3b.zip`. **Output:** `hyperkt_stage8a.zip`.

This is an exploratory validation-only ablation. **Do not evaluate the held-out test set in this notebook.** It compares four independently trained models with the same seed, epochs, data, and validation selection. Results are not five-seed statistics. No result is assumed in advance.


In [ ]:
import os, io, json, zipfile, random, copy, shutil, platform
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from google.colab import files
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Runtime:',platform.python_version(),torch.__version__,DEVICE,torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
uploaded=files.upload()
required=['hyperkt_stage2.zip','hyperkt_stage3b.zip']
assert all(n in uploaded for n in required), f'Upload {required}'
z2=zipfile.ZipFile(io.BytesIO(uploaded[required[0]]));z3=zipfile.ZipFile(io.BytesIO(uploaded[required[1]]))
def load_npz(z,n):
    with np.load(io.BytesIO(z.read(n)),allow_pickle=False) as d:return {k:d[k].copy() for k in d.files}
train=load_npz(z2,'train_sequences.npz');val=load_npz(z2,'val_sequences.npz')
hg=load_npz(z3,'hypergraph_repaired.npz');co=load_npz(z2,'train_cooccurrence.npz')
print('cooccurrence keys',list(co));print('hypergraph keys',list(hg))


In [ ]:
# Preflight checks: split separation, target shifting, padding and cross-window target counts.
assert set(train['user_id']).isdisjoint(set(val['user_id']))
for name,d in [('train',train),('validation',val)]:
    m=d['mask'];assert m.dtype==np.bool_ and m.shape[1]==200
    assert np.all((d['target_correct'][m]==0)|(d['target_correct'][m]==1))
    assert np.all((d['target_skill'][m]>=2)&(d['target_skill'][m]<=150))
    assert np.all(d['history_skill'][m]!=1)
    assert np.all(np.diff(m.astype(np.int8),axis=1)<=0),'Masks must be prefix-true, suffix-false'
    assert np.all(d['target_skill'][:,:-1][m[:,:-1]&m[:,1:]]==d['history_skill'][:,1:][m[:,:-1]&m[:,1:]])
    assert np.all(d['target_correct'][:,:-1][m[:,:-1]&m[:,1:]]==d['history_correct'][:,1:][m[:,:-1]&m[:,1:]])
    print(name,'windows',len(m),'targets',int(m.sum()),'users',len(set(d['user_id'])))
assert np.array_equal(hg['skill_ids'],np.arange(2,151))
assert hg['assignments'].shape==(149,32)
print('PASS: static preflight. Note: across-window chronological uniqueness requires original row IDs.')


In [ ]:
PAD=1;N_SKILLS=151;MAXLEN=200
A=np.zeros((N_SKILLS,32),dtype=np.float32)
A[hg['skill_ids'].astype(int)]=hg['assignments'].astype(np.float32)
A[PAD]=0;A=torch.tensor(A)
def dataset(d):
    names=['history_skill','history_correct','target_skill','target_correct','mask']
    return TensorDataset(*(torch.tensor(d[k],dtype=torch.bool if k=='mask' else torch.long) for k in names))
train_ds=dataset(train);val_ds=dataset(val)
# Frozen hypergraph embeddings: assignments -> 32-dimensional structural feature per skill
A=np.zeros((N_SKILLS,32),dtype=np.float32)
A[hg['skill_ids']]=hg['assignments'].astype(np.float32)
A[PAD]=0
A=torch.tensor(A)
class HyperKTTemporal(nn.Module):
    def __init__(self, use_hypergraph=False, use_decay=True, d=256,heads=8,layers=4,dropout=.1):
        super().__init__();self.use_hypergraph=use_hypergraph;self.use_decay=use_decay
        self.skill=nn.Embedding(N_SKILLS,d,padding_idx=PAD)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAXLEN,d)
        self.decay_raw=nn.Parameter(torch.tensor(-2.97)) # softplus ~0.05
        if use_hypergraph:
            self.register_buffer('assignments',A.clone())
            self.hg_proj=nn.Linear(32,d,bias=False)
            self.hg_gate=nn.Linear(2*d,d)
        enc=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=dropout,batch_first=True,activation='gelu',norm_first=True)
        self.encoder=nn.TransformerEncoder(enc,num_layers=layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d)
        self.head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(dropout),nn.Linear(d,1))
    def forward(self,hs,hc,ts,mask):
        B,L=hs.shape
        pos=torch.arange(L,device=hs.device)
        # Recent positions are given stronger positional weighting. No real-time gaps exist in Stage 2.
        age=(L-1-pos).float()
        decay=torch.exp(-torch.nn.functional.softplus(self.decay_raw)*age/MAXLEN) if self.use_decay else torch.ones_like(age)
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)[None,:,:]*decay[None,:,None]
        if self.use_hypergraph:
            structure=self.hg_proj(self.assignments[hs]);gate=torch.sigmoid(self.hg_gate(torch.cat([x,structure],dim=-1)))
            x=x+gate*structure
        # Prevent looking ahead at later interaction outcomes
        causal=torch.triu(torch.ones(L,L,device=hs.device,dtype=torch.bool),diagonal=1)
        # Stage 2 may pad the tail; zero padded states and rely on causal masking.
        x=x.masked_fill(~mask.unsqueeze(-1),0.0)
        h=self.norm(self.encoder(x,mask=causal))
        target=self.skill(ts)
        if self.use_hypergraph:target=target+self.hg_proj(self.assignments[ts])
        return self.head(torch.cat([h,target],dim=-1)).squeeze(-1)


In [ ]:
# Causal non-interference: changing future interaction responses must not affect earlier logits.
probe=HyperKTTemporal(use_hypergraph=True,use_decay=True,d=256,heads=8,layers=4).to(DEVICE).eval()
d=val;hs=torch.tensor(d['history_skill'][:2],device=DEVICE).long();hc=torch.tensor(d['history_correct'][:2],device=DEVICE).long()
ts=torch.tensor(d['target_skill'][:2],device=DEVICE).long();m=torch.tensor(d['mask'][:2],device=DEVICE)
cut=15
with torch.no_grad():
    original=probe(hs,hc,ts,m)
    changed=hc.clone();changed[:,cut+1:]=1-changed[:,cut+1:]
    modified=probe(hs,changed,ts,m)
delta=(original[:,:cut+1]-modified[:,:cut+1]).abs().max().item()
assert delta<1e-5,f'Causal noninterference failed: {delta}'
print('PASS: causal response perturbation, max earlier-logit delta:',delta)
del probe


In [ ]:
BATCH_SIZE=16;EPOCHS=12;PATIENCE=4;LR=2e-4
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=DEVICE.type=='cuda')
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=DEVICE.type=='cuda')
def evaluate(model,loader,optimizer=None):
    training=optimizer is not None;model.train(training)
    total_loss=0.;count=0;ys=[];ps=[]
    for batch in loader:
        hs,hc,ts,y,m=[t.to(DEVICE,non_blocking=True) for t in batch]
        if training:optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(hs,hc,ts,m)
            loss=F.binary_cross_entropy_with_logits(logits[m],y[m].float())
            if training:
                loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()
        n=int(m.sum());total_loss+=float(loss.detach())*n;count+=n
        if not training:
            ys.append(y[m].cpu().numpy());ps.append(torch.sigmoid(logits[m]).cpu().numpy())
    result={'nll':total_loss/count,'n_targets':count}
    if not training:
        y=np.concatenate(ys);p=np.concatenate(ps)
        result['auc']=float(roc_auc_score(y,p));result['brier']=float(brier_score_loss(y,p))
    return result


In [ ]:
# Independent training from initialization; same budget and seed for every variant.
# Early stopping on validation only. Hypergraph assignments remain frozen as in Stage 4.
variants={
 'transformer_no_decay':dict(use_hypergraph=False,use_decay=False),
 'transformer_decay':dict(use_hypergraph=False,use_decay=True),
 'hypergraph_no_decay':dict(use_hypergraph=True,use_decay=False),
 'hypergraph_decay':dict(use_hypergraph=True,use_decay=True),
}
out='/content/hyperkt_stage8a';os.makedirs(out,exist_ok=True)
reports={};histories=[]
for name,config in variants.items():
    print('TRAINING:',name,flush=True)
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
    model=HyperKTTemporal(**config).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    best_auc=-1.;best_epoch=0;best_state=None;wait=0
    for epoch in range(1,EPOCHS+1):
        tr=evaluate(model,train_loader,opt);va=evaluate(model,val_loader)
        histories.append({'model':name,'epoch':epoch,'train_nll':tr['nll'],**{'val_'+k:v for k,v in va.items()}})
        print(name,epoch,'AUC',round(va['auc'],6),'NLL',round(va['nll'],6),flush=True)
        if va['auc']>best_auc+1e-6:
            best_auc=va['auc'];best_epoch=epoch;best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=PATIENCE:break
    model.load_state_dict(best_state)
    selected=evaluate(model,val_loader)
    reports[name]={'best_epoch':best_epoch,'validation':selected,'configuration':config}
    torch.save({'state_dict':best_state,'config':config,'seed':SEED,'selected_epoch':best_epoch},f'{out}/{name}.pt')
    del model,opt
    if torch.cuda.is_available():torch.cuda.empty_cache()
print(json.dumps(reports,indent=2))


In [ ]:
pd.DataFrame(histories).to_csv(f'{out}/training_history.csv',index=False)
with open(f'{out}/report.json','w') as f:json.dump(reports,f,indent=2)
with open(f'{out}/protocol.txt','w') as f:
    f.write('Exploratory single-seed ablation, ASSISTments 2009, seed 42; student-disjoint train/validation; 12 max epochs; patience 4; 4-layer 8-head 256-d Transformer; validation AUC selection; no test read. Stage 2 uses positional decay, not timestamp-based forgetting. Hypergraph assignments frozen. Causal response perturbation checked. Cross-window chronological uniqueness not independently verified.\n')
shutil.make_archive('/content/hyperkt_stage8a','zip',out)
print('Output: /content/hyperkt_stage8a.zip')
files.download('/content/hyperkt_stage8a.zip')


## Interpretation
Compare **transformer_no_decay → transformer_decay** for the positional-decay contribution and **transformer_decay → hypergraph_decay** for the frozen hypergraph contribution. These comparisons are exploratory, not statistically significant without multiple independent seeds. The current architecture's decay scales positional embeddings; it is **not** a verified elapsed-time forgetting mechanism. The structural component uses fixed assignments and is not end-to-end learned here. Do not report final test results or claim causal prerequisites from this experiment.